In [ ]:
""" RESIDUAL AUGMENTED NEURAL NETWORK TRAINING"""
import torch
from torch import nn
from torch.utils.data import random_split, DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from torch.nn import functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transform = v2.Compose([
    v2.ToImage(),
    v2.TrivialAugmentWide(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.3444, 0.3809, 0.4082],
        std=[0.2038, 0.1365, 0.1155]
    )
])

test_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.3444, 0.3809, 0.4082],
        std=[0.2038, 0.1365, 0.1155]
    )
])

train_dataset = datasets.EuroSAT(
    root="data",
    download=True,
    transform= train_transform
)

test_dataset = datasets.EuroSAT(
    root="data",
    download=True,
    transform= test_transform
)

train_size = int(0.8 * len(train_dataset))
test_size = len(train_dataset) - train_size

indices = torch.randperm(len(train_dataset))

train_indices = indices[:train_size]
test_indices = indices[train_size:]

from torch.utils.data import Subset
training_data = Subset(train_dataset, train_indices)
test_data = Subset(test_dataset, test_indices)


train_dataloader = DataLoader(training_data, batch_size=64, shuffle = True, num_workers=2, pin_memory=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle = True, num_workers=2, pin_memory=True)
print(train_dataset.classes)
class Residual(nn.Module):
    """The Residual block of ResNet models."""
    def __init__(self, num_channels = 32, use_1x1conv=True, strides=1):
        super().__init__()
        self.conv1 = nn.LazyConv2d(num_channels, kernel_size=3, padding=1,
                                   stride=strides)
        self.conv2 = nn.LazyConv2d(num_channels*2, kernel_size=3, padding=1,
                                   stride=strides)
        self.conv3 = nn.LazyConv2d(num_channels*4, kernel_size=3, padding=1,
                                   stride=strides)
        self.conv4 = nn.LazyConv2d(num_channels*8, kernel_size=3, padding=1,
                                   stride=strides)
        if use_1x1conv:
            self.conv5 = nn.LazyConv2d(num_channels*2, kernel_size=1,
                                       stride=2)
            self.conv6 = nn.LazyConv2d(num_channels*8, kernel_size=1,
                                       stride= 2)
        else:
            self.conv5 = None
            self.conv6 = None
        self.bn12 = nn.LazyBatchNorm2d()
        self.bn22 = nn.LazyBatchNorm2d()
        self.bn32 = nn.LazyBatchNorm2d()
        self.bn42 = nn.LazyBatchNorm2d()
        self.bn2 = nn.LazyBatchNorm1d()
        self.bn3 = nn.LazyBatchNorm1d()
        self.resnet_block = nn.Sequential(
            self.conv1,
            self.bn12,
            nn.ReLU(),
            self.conv2,
            self.bn22,
            nn.ReLU(),
            nn.MaxPool2d(kernel_size= 2, stride=2, padding = 0)
        )
        self.resnet_block2 = nn.Sequential(
            self.conv3,
            self.bn32,
            nn.ReLU(),
            self.conv4,
            self.bn42,
            nn.ReLU(),
            nn.MaxPool2d(kernel_size= 2, stride=2, padding = 0))
        self.lin = nn.Sequential(
            nn.AdaptiveMaxPool2d((4,4)),
            nn.Flatten(),
            nn.LazyLinear(512),
            self.bn2,
            nn.ReLU(),
            nn.Linear(512, 256),
            self.bn3,
            nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, X):
        Y = self.resnet_block(X)
        X1 = X
        X2 = X
        if self.conv5 is not None:
            X1 = self.conv5(X)
        Y += X1
        Y = F.relu(Y)
        Y2 = Y
        Y = self.resnet_block2(Y)
        if self.conv6 is not None:
            X2 = self.conv6(Y2)
        Y += X2
        Y = F.relu(Y)
        logits = self.lin(Y)
        return logits

model = Residual().to(device)
x = torch.randn(2, 3, 64, 64).to(device)
print(model(x).shape)
learning_rate = (0.5)
batch_size = 64
epochs = 100
max_correct = 0

def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device)
        y = y.to(device)
        # Compute prediction and loss
        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
    scheduler.step()


def test_loop(dataloader, model, loss_fn):
    global max_correct
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    ''' if correct > max_correct:
        torch.save(model, 'Build1.pth')
        max_correct = correct
 '''
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=1e-4)
from torch.optim.lr_scheduler import (
    StepLR,
    MultiStepLR,
    ExponentialLR,
    CosineAnnealingLR,
    OneCycleLR,
    ReduceLROnPlateau
)
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=epochs,
    eta_min=1e-5
)
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
    print(str(optimizer.param_groups[0]["lr"])+"\n")
print("Done!")
print(max_correct)

torch.save(model, 'Build1.pth')


['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
torch.Size([2, 10])
Epoch 1
-------------------------------
loss: 2.424489  [   64/21600]


KeyboardInterrupt: 

In [ ]:
""" RESIDUAL AUGMENTED NETWORK PREDICTION"""
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from pathlib import Path
import csv
from PIL import Image
import torch
from torch import nn
from torch.utils.data import random_split, DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from torch.nn import functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.3444, 0.3809, 0.4082],
        std=[0.2038, 0.1365, 0.1155]
    )
])

test_data = datasets.EuroSAT(
    root="data",
    download=True,
    transform= test_transform
)

test_dataloader = DataLoader(test_data, batch_size=100, shuffle = True)

class Residual(nn.Module):
    """The Residual block of ResNet models."""
    def __init__(self, num_channels = 32, use_1x1conv=True, strides=1):
        super().__init__()
        self.conv1 = nn.LazyConv2d(num_channels, kernel_size=3, padding=1,
                                   stride=strides)
        self.conv2 = nn.LazyConv2d(num_channels*2, kernel_size=3, padding=1,
                                   stride=strides)
        self.conv3 = nn.LazyConv2d(num_channels*4, kernel_size=3, padding=1,
                                   stride=strides)
        self.conv4 = nn.LazyConv2d(num_channels*8, kernel_size=3, padding=1,
                                   stride=strides)
        if use_1x1conv:
            self.conv5 = nn.LazyConv2d(num_channels*2, kernel_size=1,
                                       stride=2)
            self.conv6 = nn.LazyConv2d(num_channels*8, kernel_size=1,
                                       stride= 2)
        else:
            self.conv5 = None
            self.conv6 = None
        self.bn12 = nn.LazyBatchNorm2d()
        self.bn22 = nn.LazyBatchNorm2d()
        self.bn32 = nn.LazyBatchNorm2d()
        self.bn42 = nn.LazyBatchNorm2d()
        self.bn2 = nn.LazyBatchNorm1d()
        self.bn3 = nn.LazyBatchNorm1d()
        self.resnet_block = nn.Sequential(
            self.conv1,
            self.bn12,
            nn.ReLU(),
            self.conv2,
            self.bn22,
            nn.ReLU(),
            nn.MaxPool2d(kernel_size= 2, stride=2, padding = 0)
        )
        self.resnet_block2 = nn.Sequential(
            self.conv3,
            self.bn32,
            nn.ReLU(),
            self.conv4,
            self.bn42,
            nn.ReLU(),
            nn.MaxPool2d(kernel_size= 2, stride=2, padding = 0))
        self.lin = nn.Sequential(
            nn.AdaptiveMaxPool2d((4,4)),
            nn.Flatten(),
            nn.LazyLinear(512),
            self.bn2,
            nn.ReLU(),
            nn.Linear(512, 256),
            self.bn3,
            nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, X):
        Y = self.resnet_block(X)
        X1 = X
        X2 = X
        if self.conv5 is not None:
            X1 = self.conv5(X)
        Y += X1
        Y = F.relu(Y)
        Y2 = Y
        Y = self.resnet_block2(Y)
        if self.conv6 is not None:
            X2 = self.conv6(Y2)
        Y += X2
        Y = F.relu(Y)
        logits = self.lin(Y)
        return logits

model = torch.load("Build1 (1).pth", weights_only= False)
model = model.to(device)
model.eval()

def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

loss_fn = nn.CrossEntropyLoss()
t = []
for i in range(4000):
  t.append("img_"+"0"*(5-len(str(i)))+str(i)+".jpg")
with open("Build1 Predictions.csv", "w") as file:
    file.write("img_id,label\n")
for i in t:
  x = Path("drive/MyDrive/Images2/"+i)
  img = Image.open(x)
  img_tensor = test_transform(img)
  img_tensor = img_tensor.unsqueeze(0).to(device)

  with torch.no_grad():
    pred = model(img_tensor)

  prediction = pred.argmax(1).item()

  with open("Build1 Predictions.csv", "a") as file:
    file.write(i+","+str(prediction)+"\n")






for t in range(1):
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Test Error: 
 Accuracy: 99.5%, Avg loss: 0.015722 

Done!
